# 07.3 - N-grams

**Phase:** 07 - NLP

**Status:** VERIFIED

---

## 1. What Are We Solving?

'not good' has the opposite meaning of 'good'. A unigram model sees two independent words; a bigram model sees one meaningful phrase. **N-grams** are contiguous sequences of N tokens that capture local word order without neural networks.

## 2. Why Does This Matter?

BoW/TF-IDF discard sequence information entirely. N-grams restore a little: negation ('not good'), domain phrases ('machine learning'), and idioms. They are cheap, interpretable, and still widely used.

## 3. Prerequisites

- Unit 07.2 (BoW/TF-IDF)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Extract unigrams, bigrams, trigrams manually and with sklearn
- Explain what N-grams capture that unigrams miss
- Use `ngram_range` and `min_df`/`max_features` to control the feature space
- Combine N-grams with TF-IDF effectively

## 5. Mental Model

A sliding window of size N over a tokenized document.

```text
Text: "the cat sat on the mat"
Bigrams:  [the cat, cat sat, sat on, on the, the mat]
Trigrams: [the cat sat, cat sat on, sat on the, on the mat]
```


## 6. Setup


In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

print('Setup OK')


Setup OK


## 7. Manual N-gram Extraction

Build N-grams from scratch with a sliding window.


In [2]:
def ngrams(tokens, n):
    return [' '.join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

text = "the cat sat on the mat"
tokens = text.split()
print("Unigrams :", ngrams(tokens, 1))
print("Bigrams  :", ngrams(tokens, 2))
print("Trigrams :", ngrams(tokens, 3))

# Demonstrate negation capture with bigrams
s = "This movie is not good".lower().split()
print("\nNegation bigrams:", [b for b in ngrams(s, 2) if 'not' in b])


Unigrams : ['the', 'cat', 'sat', 'on', 'the', 'mat']
Bigrams  : ['the cat', 'cat sat', 'sat on', 'on the', 'the mat']
Trigrams : ['the cat sat', 'cat sat on', 'sat on the', 'on the mat']

Negation bigrams: ['is not', 'not good']


## 8. N-gram Feature Space Explodes

The number of possible N-grams grows quickly with N. Watch vocabulary size.


In [3]:
docs = [
    "I do not like spam at all",
    "I like good fresh food",
    "spam is not good for health",
    "fresh food is always good",
]

for n in [1, 2, 3, 4]:
    v = CountVectorizer(ngram_range=(n, n))
    M = v.fit_transform(docs)
    print(f"{n}-gram: vocab={len(v.vocabulary_):4d}  features={M.shape[1]:4d}  (dense size ~{M.shape[0]*M.shape[1]})")

print("\nVocabulary grows only modestly here (tiny corpus), but on large corpora it explodes.")


1-gram: vocab=  13  features=  13  (dense size ~52)
2-gram: vocab=  16  features=  16  (dense size ~64)
3-gram: vocab=  13  features=  13  (dense size ~52)
4-gram: vocab=   9  features=   9  (dense size ~36)

Vocabulary grows only modestly here (tiny corpus), but on large corpora it explodes.


## 9. Unigrams + Bigrams with TF-IDF

The industry default: `ngram_range=(1, 2)` with TF-IDF and stopword removal.


In [4]:
vec = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')
X = vec.fit_transform(docs)
feats = vec.get_feature_names_out()

print("Length of feature list:", len(feats))
print("Feature sample:", feats[:10])

# Show single-word and two-word features coexist
unigram = [f for f in feats if ' ' not in f]
bigram = [f for f in feats if ' ' in f]
print(f"Unigrams: {len(unigram)}, Bigrams: {len(bigram)}")


Length of feature list: 13
Feature sample: ['food' 'food good' 'fresh' 'fresh food' 'good' 'good fresh' 'good health'
 'health' 'like' 'like good']
Unigrams: 6, Bigrams: 7


## 10. Highest-Ranked Bigrams

Inspect the most informative N-grams for interpretation.


In [5]:
import numpy as np
max_row = np.asarray(X.max(axis=0).toarray()).ravel()
order = np.argsort(max_row)[::-1]
print("Top 10 N-gram features by max TF-IDF score:")
for i in order[:10]:
    print(f"  {feats[i]:20s} score={max_row[i]:.3f}")


Top 10 N-gram features by max TF-IDF score:
  like spam            score=0.668
  food good            score=0.553
  like                 score=0.526
  spam                 score=0.526
  spam good            score=0.498
  good health          score=0.498
  health               score=0.498
  like good            score=0.452
  good fresh           score=0.452
  fresh                score=0.436


## 11. N-grams Catch Negation

Show why 'not good' bigram matters for sentiment vs 'good' unigram.


In [6]:
probe = CountVectorizer(ngram_range=(1, 2), stop_words=None)
P = probe.fit_transform(docs)
vocab = {w: i for i, w in enumerate(probe.get_feature_names_out())}

def contains(doc, feat):
    if feat not in vocab:
        return 0
    return int(P[doc, vocab[feat]] > 0)

print("'good' appears in docs:", [contains(i, 'good') for i in range(4)])
print("'not good' appears in docs:", [contains(i, 'not good') for i in range(4)])
print("\nDoc 0 & 2 contain 'not good' — a bigram the unigram model would misread.")


'good' appears in docs: [0, 1, 1, 1]
'not good' appears in docs: [0, 0, 1, 0]

Doc 0 & 2 contain 'not good' — a bigram the unigram model would misread.


## 12. Failure Case: N Too Large / Unbounded Vocabulary

High N and no limits -> sparse, overfit features and memory blowups.


In [7]:
long_corpus = []
for k in range(30):
    long_corpus.append("word_a word_b word_c word_d word_e word_f word_g ".split()[:k+1] and " ".join([f"w{k}_{i}" for i in range(20)]))

for n in [2, 3, 5]:
    v = CountVectorizer(ngram_range=(n, n), min_df=1)
    M = v.fit_transform(long_corpus)
    print(f"ngram_range=({n},{n}): vocab={len(v.vocabulary_)}")

print("\nRecommendation: keep N<=3 and cap with max_features/min_df.")


ngram_range=(2,2): vocab=570
ngram_range=(3,3): vocab=540
ngram_range=(5,5): vocab=480

Recommendation: keep N<=3 and cap with max_features/min_df.


## 13. Debugging: Common Errors

- **Vocabulary millions** — N too large, no `max_features`. Fix: N<=3, cap size.
- **Overfitting with N-grams** — too many sparse features. Fix: reduce range, raise `min_df`.
- **Key phrases missing** — `ngram_range` too narrow. Fix: use `(1,2)` or `(1,3)`.
- **Memory error** — dense matrix. Fix: keep sparse.

## 14. Real-World Considerations

- Spam filters use 'free money', 'click here' bigrams.
- Sentiment uses negation bigrams. Autocomplete uses N-grams for next-word.
- Always combine with `max_features`/`min_df`.

## 15. Common Mistakes

- Using N > 3.
- Dropping unigrams when using bigrams.
- Tokenizing (or not) inconsistently so punctuation becomes part of phrases.

## 16. When NOT to Use

- Very large vocabularies / long documents (prefer subword or embeddings).
- When you need true semantic similarity.

## 17. Challenge

Compare character-level N-grams (2-5) vs word-level N-grams on typos.


In [8]:
# Challenge: char-level n-grams handle typos well
typo_docs = ["machine learnning is fun", "I love machin learning",
             "deep learnig is powerful", "ml is fun and useful"]

cv_char = CountVectorizer(analyzer='char_wb', ngram_range=(2, 4), max_features=30)
M = cv_char.fit_transform(typo_docs)
print("Char n-gram vocab sample:", cv_char.get_feature_names_out()[:12])
print("\nTypos like 'learnning'/'learnig' still share char substrings"
      " e.g. 'earn', 'lern'), so char n-grams stay robust.")


Char n-gram vocab sample: [' i' ' is' ' is ' ' l' ' le' ' lea' ' m' ' mac' 'ac' 'ar' 'arn' 'arni']

Typos like 'learnning'/'learnig' still share char substrings e.g. 'earn', 'lern'), so char n-grams stay robust.


## 18. Closed-Book Recall

1. Give an N-gram that a unigram model cannot represent.
2. Why does vocabulary explode as N grows?
3. What is the default recommended `ngram_range` for classical ML?
4. Name a real application using N-grams.

## 19. Teach-Back Questions

- Explain how a bigram captures negation.
- Show how you'd control an N-gram vocabulary that is too large.

## 20. Summary

You extracted N-grams manually and with sklearn, combined them with TF-IDF, inspected top features, and saw how they restore local word order.

## 21. Further Experiment

- Tune `ngram_range` as a hyperparameter in a classifier.
- Try `(1,3)` with `min_df=2` to reduce noise.

## 22. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, scikit-learn, matplotlib
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
